<div align="center">

# Week 04: Sensors and Actuators: Sampling, Quantisation, Noise and Smart Meters

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-04) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/Week04_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the constants of the measurements of the campus: The grid voltage, the noise of the frequency meter and the meter of the library. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The measurements of the campus microgrid
F0_HZ, V_PEAK = 50.0, 325.0                  # the grid voltage: 50 Hz, 230 V rms
SIGMA_F_HZ, FS_F_HZ = 0.025, 100.0           # noise of one frequency reading, readings a second
METER_IMP_KWH, CT_RATIO = 1000, 200 / 5      # pulses of the meter per kWh, current transformer 200/5 A
P_BASE_KW, P_HEAT_KW = 20.0, 60.0            # the library in the evening: base load and heater
print("Setup complete. Running in Colab:", IN_COLAB, "| grid", F0_HZ, "Hz | frequency noise", SIGMA_F_HZ,
      "Hz | meter", METER_IMP_KWH, "imp/kWh with CT ratio", CT_RATIO)

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 4.1": (lambda: 20, 1e-9),
    "Exercise 4.2": (lambda: 12.5, 1e-9),
    "Exercise 4.3": (lambda: 6.0, 1e-9),
    "Exercise A": (lambda: gain * 891 + offset, 1e-6),
    "Exercise B": (lambda: 20.0, 1e-6),
    "Exercise C": (lambda: 1000 * (10 / 2 ** 10) / math.sqrt(12), 1e-4),
    "Exercise D": (lambda: 60.0, 1e-6),
    "Exercise E": (lambda: float(STAGE4['window for a frequency noise below 10 mHz (samples)']), 1e-9),
}

## Python step 4: Signals as arrays

A measured signal is a sequence of numbers taken at regular times. This step builds the tools of the week. It samples a signal into an array, rounds it to the steps of a converter, adds random noise and smooths it with a moving average &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "P_HEAT_KW" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Sampling a signal

In [ ]:
# Python practice: a small example for learning the language
t = np.arange(0, 0.02, 0.001)              # one period of 20 ms in steps of 1 ms
v = 325 * np.sin(2 * np.pi * 50 * t)         # the grid voltage in volts
print("samples:", len(t))
print(np.round(v[:6], 1))

`np.arange(0, 0.02, 0.001)` gives the sampling times, and `np.sin` turns them into the voltage at each time. Twenty samples cover one period of 50 Hz when one sample is taken every millisecond.

### Rounding and quantisation

In [ ]:
# Python practice: a small example for learning the language
q = 20.0                                     # one step of a coarse converter, in volts
vq = q * np.round(v / q)
print(np.round(vq[:6], 1))
print("largest error:", round(float(np.max(np.abs(vq - v))), 2), "V")

Dividing by the step, rounding and multiplying back gives the nearest level of the converter. The error never exceeds half a step, here 10 V.

### Random noise

In [ ]:
# Python practice: a small example for learning the language
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(0.0, 5.0, size=1000)      # mean 0 V, standard deviation 5 V
print("mean:", round(float(noise.mean()), 3), "| standard deviation:", round(float(noise.std()), 3))

`rng.normal(mean, std, size)` draws normally distributed numbers. A fixed seed makes the draws repeatable, so every run of the Colab notebook gives the same numbers.

### Moving averages

In [ ]:
# Python practice: a small example for learning the language
x = np.array([1.0, 3.0, 2.0, 6.0, 4.0, 5.0])
print(np.convolve(x, np.ones(3) / 3, mode="valid"))

`np.convolve` with a window of three equal weights averages each group of three neighbours. The option `mode="valid"` keeps only the windows that lie completely inside the signal.

**Quick check.** Why does the moving average above return four values for six inputs?

<details><summary>Answer</summary>

A window of three fits into six values only four times; the result has N - 1 = 2 values fewer.

</details>

### Exercises for Python step 4

**Exercise 4.1.** Store the number of samples in one period of 50 Hz at a sampling rate of 1000 samples a second in `n_period`.

In [ ]:
n_period = None   # your answer

check("Exercise 4.1", n_period)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_period = len(np.arange(0, 1 / 50, 1 / 1000))

check("Exercise 4.1", n_period)

**Exercise 4.2.** Quantise the value 12.34 with a step of 0.5 and store the result in `x_q`.

In [ ]:
x_q = None   # your answer

check("Exercise 4.2", x_q)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
x_q = 0.5 * round(12.34 / 0.5)

check("Exercise 4.2", x_q)

**Exercise 4.3.** Average the values `[3, 5, 4, 8]` with a window of two using `np.convolve` and store the last value of the result in `last_avg`.

In [ ]:
last_avg = None   # your answer

check("Exercise 4.3", last_avg)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
last_avg = float(np.convolve([3, 5, 4, 8], np.ones(2) / 2, mode="valid")[-1])

check("Exercise 4.3", last_avg)

## 1. Sensors and calibration &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#ref-1)&#93;

*Lecture: [Sensors: From a physical quantity to a number](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#sensors)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "noise" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A sensor turns a physical quantity into a voltage, and a converter turns the voltage into a number. Python follows a temperature sensor of the library through this chain and corrects its errors with a calibration at two known temperatures.

**From the formulas to the code.** The converter $c = \mathrm{round}(V / V_{ref} \cdot (2^n - 1))$ is `adc(v, bits)`. The two-point calibration $T = a\,c + b$ with $a = (T_2 - T_1)/(c_2 - c_1)$ and $b = T_1 - a\,c_1$ gives `gain` and `offset`.

In [ ]:
def adc(v, bits=12, v_ref=3.3):
    """An ideal analog-to-digital converter from 0 to v_ref with 2**bits levels."""
    return np.clip(np.round(np.asarray(v, float) / v_ref * (2 ** bits - 1)), 0, 2 ** bits - 1).astype(int)

def sensor_volts(T):
    """The temperature sensor of the library: nominally 0.5 V at 0 C and 10 mV per kelvin."""
    return 0.512 + 0.0098 * np.asarray(T, float)   # its true behaviour, unknown to the controller

c1, c2 = adc(sensor_volts([0.0, 50.0]))            # an ice bath and a warm bath of 50 C
gain = (50.0 - 0.0) / (c2 - c1)
offset = 0.0 - gain * c1
print(f"calibration: {c1} and {c2} counts, gain {gain:.5f} K per count, offset {offset:.2f} C")
for T in [10.0, 21.0, 30.0]:
    c = int(adc(sensor_volts(T)))
    nominal = (c * 3.3 / 4095 - 0.5) / 0.01         # with the values of the data sheet
    print(f"T = {T:4.1f} C: {c} counts, data sheet {nominal:6.2f} C, calibrated {gain * c + offset:6.2f} C")

**Python note.** `np.clip(x, lo, hi)` keeps the counts inside the range of the converter, and `.astype(int)` turns them into integers. The function `sensor_volts` stands for the real sensor, whose small errors only a calibration reveals.

**Exercise A.** Store the calibrated temperature for a reading of 891 counts in `T_a`.

In [ ]:
T_a = None   # your answer

check("Exercise A", T_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T_a = gain * 891 + offset

check("Exercise A", T_a)

In [ ]:
# Figure: the temperature from the counts with the data sheet and after the calibration
T = np.linspace(-5, 55, 61)
c = adc(sensor_volts(T))
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(T, (c * 3.3 / 4095 - 0.5) / 0.01 - T, color="#B53F2E", label="data sheet")
ax.plot(T, gain * c + offset - T, color="#2E8B57", label="after the calibration")
ax.axhline(0, color="#56616D", lw=1)
ax.set_xlabel("true temperature (C)")
ax.set_ylabel("error of the reading (K)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why does the error of the data sheet grow with the temperature? Why is the calibrated error never zero?

## 2. Sampling and aliasing: Your measurements &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#ref-3)&#93;

*Lecture: [Sampling: How often to look](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#sampling)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "offset" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A computer sees a signal only at its sampling instants. Python shows that a sine sampled too slowly appears at a wrong, lower frequency, and checks your measurements on the sampling bench against the formula.

**From the formulas to the code.** Shannon's condition $f_s > 2 f_{max}$ decides whether a signal can be reconstructed. The apparent frequency $f_a = |f - k f_s|$, with $k$ the integer nearest to $f/f_s$, is `alias(f, fs)`. `apparent_from_samples` counts the rising zero crossings of the samples, as a simple frequency meter does.

In [ ]:
def alias(f, fs):
    """The apparent frequency of a sine of f Hz sampled at fs Hz, between 0 and fs / 2."""
    return abs(f - fs * round(f / fs))

def apparent_from_samples(f, fs, seconds=10.0, phase=0.3):
    n = np.arange(int(seconds * fs))
    x = np.sin(2 * np.pi * f * n / fs + phase)
    i = np.where((x[:-1] < 0) & (x[1:] >= 0))[0]            # rising zero crossings
    t = (i - x[i] / (x[i + 1] - x[i])) / fs                  # crossing times, interpolated between samples
    return (len(t) - 1) / (t[-1] - t[0])

for fs in [1000, 200, 110, 90, 70, 60]:
    print(f"fs = {fs:>4} Hz: 50 Hz appears at {alias(50, fs):5.1f} Hz, counted {apparent_from_samples(50, fs):5.1f} Hz"
          + ("  (aliased)" if fs <= 2 * 50 else ""))

**Python note.** `round(f / fs)` gives the integer $k$ of the formula. The comparison `(x[:-1] < 0) & (x[1:] >= 0)` is `True` where one sample is negative and the next one is not, which marks a rising zero crossing. The crossing times are interpolated between the two samples, and the frequency is the number of periods between the first and the last crossing divided by their distance.

**Exercise B.** Store the apparent frequency of the 50 Hz voltage sampled at 70 Hz in `f_b`.

In [ ]:
f_b = None   # your answer

check("Exercise B", f_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
f_b = alias(50, 70)

check("Exercise B", f_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lab.html) samples the voltage of the grid on a bench with a signal generator, a converter board and an oscilloscope. Paste the copied rows into `MY_SAMPLES`, and the code compares each of your measurements with the formulas.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_SAMPLES = """
"""
if MY_SAMPLES.strip():
    samples = pd.read_csv(io.StringIO(MY_SAMPLES.strip()))
    source = "your own measurements on the sampling bench"
else:
    samples = pd.DataFrame({"test": [1, 2, 3, 4, 5, 6, 7], "fs_hz": [1000, 110, 90, 60, 1000, 1000, 1000],
                            "bits": [12, 12, 12, 12, 4, 8, 10], "apparent_hz": [50.0, 50.0, 40.0, 10.0, 50.0, 50.0, 50.0],
                            "rms_err_mv": [0.7, 0.7, 0.7, 0.7, 180.4, 11.3, 2.8]})
    source = "a simulated student"
samples["model_hz"] = [alias(50, fs) for fs in samples["fs_hz"]]
samples["model_mv"] = (1000 * 10 / 2 ** samples["bits"] / math.sqrt(12)).round(1)
print("Data:", source)
print(samples.to_string(index=False))

In [ ]:
# Figure: the voltage of 50 Hz sampled at 60 Hz, and the slow sine that the samples suggest
tt = np.linspace(0, 0.2, 2000)
ts = np.arange(0, 0.2, 1 / 60)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(tt, np.sin(2 * np.pi * 50 * tt), color="#C9D1D8", lw=1, label="50 Hz")
ax.plot(ts, np.sin(2 * np.pi * 50 * ts), "o", color="#1E6FA8", label="samples at 60 Hz")
ax.plot(tt, np.sin(-2 * np.pi * 10 * tt), color="#B53F2E", lw=1.5, label="apparent 10 Hz")
ax.set_xlabel("time (s)")
ax.set_ylabel("voltage (per unit)")
ax.legend(fontsize=8, loc="upper right")
plt.show()

**Observe and reflect.** Which sampling rates on your bench gave a wrong frequency, and could the samples alone have told you? What does an analog filter before the converter do about it?

## 3. Quantisation and noise &#91;[4](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#ref-4)&#93;

*Lecture: [Quantisation and noise: How finely to look](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#quantisation)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "alias" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A converter with $n$ bits can only output $2^n$ levels, and every measurement carries some noise. Python measures the error of quantisation for several converters and shows how a moving average trades noise against delay.

**From the formulas to the code.** For the full-scale (FS) range $V_{FS}$, the loop over `bits` computes the step $q = V_{FS}/2^n$. It also computes the root-mean-square (RMS) error $q/\sqrt{12}$ and the signal-to-quantisation-noise ratio (SQNR) $6.02\,n + 1.76$ dB of a full-scale sine. Averaging $N$ independent readings divides the noise by $\sqrt{N}$ and delays the result by $(N-1)/2$ sampling periods, which the second loop checks.

In [ ]:
def quantise(x, bits, full_range):
    q = full_range / 2 ** bits
    return q * np.round(np.asarray(x) / q), q

t = np.arange(0, 1, 1 / 5000)
v = 3.25 * np.sin(2 * np.pi * 50 * t)            # the divider scales 325 V down to 3.25 V
for bits in [4, 8, 10, 12, 16]:
    vq, q = quantise(v, bits, 10.0)               # a converter with an input range from -5 V to +5 V
    err = vq - v
    print(f"{bits:>2} bits: step {1000 * q:8.3f} mV, RMS error {1000 * err.std():7.3f} mV "
          f"(q / sqrt(12) = {1000 * q / math.sqrt(12):7.3f} mV), SQNR {20 * np.log10(v.std() / err.std()):5.1f} dB")

rng = np.random.default_rng(RANDOM_STATE)
f_meas = F0_HZ + rng.normal(0, SIGMA_F_HZ, 3000)  # 30 s of frequency readings, 100 a second
for N in [1, 4, 7, 10, 25]:
    avg = np.convolve(f_meas, np.ones(N) / N, mode="valid")
    print(f"window {N:>2}: noise {1000 * avg.std():5.2f} mHz (formula {1000 * SIGMA_F_HZ / math.sqrt(N):5.2f} mHz), "
          f"delay {(N - 1) / 2 * 1000 / FS_F_HZ:5.1f} ms")

**Python note.** The error of a fine converter behaves like a uniform random number between $-q/2$ and $+q/2$, whose standard deviation is $q/\sqrt{12}$. The sine of 3.25 V uses only 65 percent of the range of 5 V. Its SQNR therefore lies 3.7 dB below the formula for a full-scale sine.

**Exercise C.** Store the RMS error of quantisation, in mV, of a 10-bit converter over a range of 10 V in `e_c`.

In [ ]:
e_c = None   # your answer

check("Exercise C", e_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
e_c = 1000 * (10 / 2 ** 10) / math.sqrt(12)

check("Exercise C", e_c)

In [ ]:
# Figure: a sine through a 4-bit converter, and the noise of the frequency against the window
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
tt = np.arange(0, 0.04, 1 / 5000)
v4, q4 = quantise(3.25 * np.sin(2 * np.pi * 50 * tt), 4, 10.0)
ax[0].plot(1000 * tt, 3.25 * np.sin(2 * np.pi * 50 * tt), color="#C9D1D8")
ax[0].step(1000 * tt, v4, color="#1E6FA8", where="mid")
ax[0].set_xlabel("time (ms)")
ax[0].set_ylabel("voltage (V)")
Ns = np.arange(1, 31)
ax[1].plot(Ns, [1000 * np.convolve(f_meas, np.ones(n) / n, mode="valid").std() for n in Ns], "o", color="#1E6FA8", ms=4)
ax[1].plot(Ns, 1000 * SIGMA_F_HZ / np.sqrt(Ns), color="#B53F2E")
ax[1].axhline(10, ls=":", color="#56616D")
ax[1].set_xlabel("window of the moving average (samples)")
ax[1].set_ylabel("noise (mHz)")
plt.tight_layout()
plt.show()

**Observe and reflect.** How many bits does the converter of the sampling bench need before the noise of the measurement, rather than the quantisation, limits the accuracy? Why can a moving average not remove a delay?

## 4. Smart meters: Your pulses &#91;[5](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#ref-5)&#93;

*Lecture: [Smart meters: Energy in intervals](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#smart-meters)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "quantise" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The meter of the library counts energy, flashes a light for every small portion of it and reports the energy of each interval of 15 minutes. Python turns the time between flashes into power and shows what the intervals hide.

**From the formulas to the code.** One pulse stands for $1/c$ kWh on the secondary side of the current transformer (CT). The primary power is therefore $P = 3600 \cdot k_{CT} / (c \cdot T_p)$ kW for a time $T_p$ in seconds between pulses, which `power_from_pulses` computes. The demand of an interval is its energy divided by 0.25 h.

In [ ]:
def power_from_pulses(period_s, imp_kwh=METER_IMP_KWH, ct=CT_RATIO):
    """Primary power in kW from the time between two pulses of the light of the meter."""
    return 3600 / (imp_kwh * period_s) * ct

print(f"pulse every 1.80 s: {power_from_pulses(1.80):.1f} kW | every 7.20 s: {power_from_pulses(7.20):.1f} kW")

def library_power(minutes, t_on=18.0, t_off=9.0):
    """The library in the evening: base load and a thermostat with a band of 0.2 K at 5 C (Week 3)."""
    phase = np.asarray(minutes) % (t_on + t_off)
    return P_BASE_KW + P_HEAT_KW * (phase < t_on)

m = np.arange(0, 240, 1 / 60)                     # four hours in steps of one second
P = library_power(m)
demand = P.reshape(-1, 15 * 60).mean(axis=1)      # the mean power of each interval of 15 minutes
print("15-min demands (kW):", np.round(demand, 1))
print(f"energy of the four hours {P.mean() * 4:.1f} kWh | highest demand {demand.max():.1f} kW | "
      f"momentary peak {P.max():.1f} kW | switchings {int(np.sum(np.abs(np.diff(P)) > 0))}")

**Python note.** `P.reshape(-1, 900)` folds the 14,400 values of one second each into rows of 900 seconds, one row for each interval of 15 minutes, and `.mean(axis=1)` averages each row. The 16 demands hide the 17 switchings of the heater. They repeat every 135 minutes, the least common multiple of the cycle of 27 minutes and the interval of 15 minutes.

**Exercise D.** Store the primary power in kW for a time of 2.4 s between pulses in `p_d`.

In [ ]:
p_d = None   # your answer

check("Exercise D", p_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_d = power_from_pulses(2.4)

check("Exercise D", p_d)

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lab.html) opens the meter cabinet of the library. Paste the copied rows into `MY_PULSES`, and the code compares your timing with the true power.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_PULSES = """
"""
if MY_PULSES.strip():
    pulses = pd.read_csv(io.StringIO(MY_PULSES.strip()))
    source = "your own pulses at the meter cabinet"
else:
    pulses = pd.DataFrame({"run": [1, 2], "heater": ["on", "off"], "pulses": [6, 4],
                           "seconds": [9.12, 21.35], "power_kw": [78.9, 20.2]})
    source = "a simulated student"
pulses["true_kw"] = np.where(pulses["heater"] == "on", P_BASE_KW + P_HEAT_KW, P_BASE_KW)
pulses["error_percent"] = (100 * (pulses["power_kw"] - pulses["true_kw"]) / pulses["true_kw"]).round(1)
print("Data:", source)
print(pulses.to_string(index=False))
on, off = pulses[pulses["heater"] == "on"], pulses[pulses["heater"] == "off"]
if len(on) and len(off):
    print(f"heater power from your pulses: {on['power_kw'].mean() - off['power_kw'].mean():.1f} kW (true {P_HEAT_KW:.0f} kW)")

**Python note.** The time of $N$ pulses spans $N - 1$ periods, so the mean period is the time divided by $N - 1$. Your reaction time spreads over all the periods, and more pulses make the error smaller.

In [ ]:
# Figure: the power of the library second by second and the demands of 15 minutes
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(m / 60, P, color="#C98510", lw=1, label="momentary power")
ax.step(np.arange(16) * 0.25, demand, where="post", color="#1E6FA8", lw=2, label="15-min demand")
ax.set_xlabel("hours after 20:00")
ax.set_ylabel("power (kW)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** What does the operator of the campus see in the interval data, and what not? Which of the two curves would you send to a controller, and which to the bill?

## 5. The campus microgrid, stage 4

*Lecture: [The campus microgrid, stage 4: Measuring the microgrid](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#the-campus-microgrid-stage-4)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "library_power" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 4 fixes the measurements that the later weeks rely on. The first result is the window that makes the frequency reading quiet enough for the controller of Week 5, with its delay. The second is the resolution of the converter, and the third is the data of the meters of the campus.

In [ ]:
STAGE4 = {}
N = math.ceil((SIGMA_F_HZ / 0.010) ** 2)                    # the noise must stay below 10 mHz
STAGE4["window for a frequency noise below 10 mHz (samples)"] = N
STAGE4["noise with this window (mHz)"] = round(1000 * SIGMA_F_HZ / math.sqrt(N), 2)
STAGE4["delay of the window (ms)"] = (N - 1) / 2 * 1000 / FS_F_HZ
STAGE4["bits for 1 mHz over 45 to 55 Hz"] = math.ceil(math.log2((55 - 45) / 0.001))
STAGE4["library: highest 15-min demand (kW)"] = round(float(demand.max()), 1)
STAGE4["library: switchings hidden in four hours"] = int(np.sum(np.abs(np.diff(P)) > 0))
meters, intervals, registers, bytes_each = 8, 96, 4, 4      # PCC, battery, six feeders
STAGE4["data of the campus meters a day (kB)"] = round(meters * intervals * registers * bytes_each / 1000, 2)
for key, value in STAGE4.items():
    print(f"{key:<52} {value}")

**Python note.** `math.ceil` rounds up, because a window of 6.25 samples does not exist and 6 would miss the target. `math.log2` gives the number of bits that $2^n$ levels need.

In [ ]:
# Figure: the frequency readings raw and through the window of stage 4
avg = np.convolve(f_meas, np.ones(N) / N, mode="valid")
ts = np.arange(len(f_meas)) / FS_F_HZ
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(ts[:300], 1000 * (f_meas[:300] - F0_HZ), color="#C9D1D8", label="raw readings")
ax.plot(ts[N - 1:300], 1000 * (avg[:300 - N + 1] - F0_HZ), color="#1E6FA8", label=f"moving average of {N}")
ax.axhline(10, ls=":", color="#B53F2E")
ax.axhline(-10, ls=":", color="#B53F2E")
ax.set_xlabel("time (s)")
ax.set_ylabel("deviation from 50 Hz (mHz)")
ax.legend(fontsize=8)
plt.show()

**Exercise E.** Store the window of the moving average for the frequency, as kept in `STAGE4`, in `n_e`.

In [ ]:
n_e = None   # your answer

check("Exercise E", n_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_e = STAGE4["window for a frequency noise below 10 mHz (samples)"]

check("Exercise E", n_e)

**Observe and reflect.** The window delays the frequency reading. Compare the delay with the margin of 1.072 s of Week 8. Which measurement of the campus would you improve first?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE4" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-04/"
               "NB04_sensors_actuators.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Every cyber-physical system starts with its sensors. The functions of this week check a sensor for its sampling rate, its resolution and its noise after one change of a name.

The sensors and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "pump_vibration"
# choose: grid_voltage, heart_rate, tyre_pressure, weather_station, pump_vibration, water_meter
APPS = {  # (signal, highest frequency of interest in Hz, sampling rate in Hz, bits, range, unit, noise of one reading)
    "grid_voltage": ("voltage of the grid", 50.0, 1000.0, 12, 700.0, "V", 0.5),
    "heart_rate": ("optical pulse sensor of a wristband", 3.0, 25.0, 12, 1.0, "a.u.", 0.004),
    "tyre_pressure": ("pressure of a car tyre", 0.01, 0.1, 10, 500.0, "kPa", 1.0),
    "weather_station": ("air temperature", 0.001, 1 / 60, 10, 80.0, "C", 0.1),
    "pump_vibration": ("vibration of a pump bearing", 400.0, 1000.0, 16, 160.0, "m/s2", 0.05),
    "water_meter": ("flow of a water meter", 0.5, 2.0, 8, 60.0, "l/min", 0.3)}
desc, f_max, fs, bits, rng_, unit, sigma = APPS[APP]
q = rng_ / 2 ** bits
print(f"{APP}: {desc}")
print(f"  sampling: {fs:g} Hz against 2 x {f_max:g} Hz -> " + ("enough" if fs > 2 * f_max else "too slow: aliasing"))
print(f"  resolution: step {q:.4g} {unit}, RMS error {q / math.sqrt(12):.4g} {unit}")
print(f"  noise {sigma:g} {unit}: " + ("the noise dominates" if sigma > q / math.sqrt(12) else "the quantisation dominates")
      + f"; averaging 16 readings gives {sigma / 4:.3g} {unit}")

**Python note.** Each entry holds the signal, the highest frequency that matters, the sampling rate, the bits, the range, the unit and the noise of one reading. The three printed lines are the checks of this week.

**Observe and reflect.** Which sensor samples too slowly, and what would you change first: The rate, the bits or the filter? Which of them would drain the battery of a wireless sensor most?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a class `Sensor` with a calibration, a converter and a filter, and test it with readings whose correct values you know. |
| Embedded systems | Write the moving average of the frequency as it would run on a microcontroller: a ring buffer of N integers and a running sum, without floating-point numbers. |
| Internet of Things | A wireless sensor sends one reading a minute and sleeps in between. How does its battery life change if it sends the average of ten readings instead? |
| Computer networks | The meters of the campus send their intervals over a link of 9600 bit/s. How long does a day of data take, and what if the meters report every minute? |
| Control engineering | Replace the moving average by the exponential filter y = y + a (x - y) and find the value of a with the same noise. Compare the delays. |
| Data science | Estimate the noise of the frequency meter from its readings alone, and test whether the readings are normally distributed. |
| Cybersecurity | A manipulated meter scales its pulses by 0.9. How many pulses would you need to detect it with the timing of Part C? |
| Artificial intelligence | Detect the switchings of the heater in the second-by-second power and estimate its power from the steps, as a non-intrusive load monitor does. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-04/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-04).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>